In [ ]:
import pandas
pandas.options.mode.chained_assignment = None

In [ ]:
# Load the "level0_sig_interactions.txt" , which is an output file from Multimorbid3D 
# You can change this to "level1_sig_interactions.txt" or "level2_sig_interactions.txt" if you like


sig_interactions = pandas.read_csv('level0_sig_interactions.txt', sep='\t')

In [ ]:
sig_interactions 

In [ ]:
#The dataframe above have not yet been filtered to show only traits that are significant after bootstrapping.. 
#So we need to filter this manually

#load the 'significant_enrichment_bootstrap.txt', which is an output file of Multimorbid3D
significant_enrichment_bootstrap = pandas.read_csv('significant_enrichment_bootstrap.txt', sep='\t')

#Lets subset the dataframe above to only show rows where sim_pval is less than or equal to 0.05
#This is the same with Supplementary Table 9
significant_enrichment_bootstrap = significant_enrichment_bootstrap[significant_enrichment_bootstrap['sim_pval'] <= 0.05]


In [ ]:
significant_enrichment_bootstrap

In [ ]:
#Store the traits that are significant at the chosen level after bootstrapping in a list
level = 'level0' # <- change the level as needed

bootstrap_significant_traits = significant_enrichment_bootstrap[significant_enrichment_bootstrap['level']==level]['trait'].unique()

In [ ]:
#Subset the 'sig_interactions' dataframe to only show rows whose traits are in the 'bootstrap_significant_traits' list

sig_interactions_bootstrap_significant = sig_interactions[sig_interactions['trait'].isin(bootstrap_significant_traits)]

In [ ]:
sig_interactions_bootstrap_significant

# Making input matrix for hierarchical clustering (used to make Figure 4)

In [ ]:
#Create a new pandas dataframe

dataframe = pandas.DataFrame({'Trait' : list(sig_interactions_bootstrap_significant["trait"].unique())})
dataframe

In [ ]:
#Make new columns with the unique genes in "level0_sig_interactions_bootstrap_significant"

gene_list = list(sig_interactions_bootstrap_significant['gene'].unique())

#Fill all cells with 0
for gene in gene_list:
    dataframe[gene] = 0

In [ ]:
dataframe

In [ ]:
#A function to fill the cells in the "dataframe" above by the number of unique GWAS SNPs of a trait that regulate different level 0 genes

def unique_gwas_eQTLs(trait, gene):
    subset = sig_interactions_bootstrap_significant[(sig_interactions_bootstrap_significant['trait']==trait) & (sig_interactions_bootstrap_significant['gene']==gene)]
    unique_gwas_eqtls = len(set(subset['rsidt']))
    return unique_gwas_eqtls

In [ ]:
#Use the function to fill the cells

for index,row in dataframe.iterrows():
    for gene in gene_list:
        dataframe.loc[index, gene] = unique_gwas_eQTLs(row[0], gene)

In [ ]:
dataframe

In [ ]:
# Save the "dataframe" as csv so we can use as an input to Pheatmap in R to make Figure 4 
dataframe.to_csv('Multimorbid3D_SNP-gene-trait_clustering.csv', index = False)
